In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bl9cksheep/medical-image")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'medical-image' dataset.
Path to dataset files: /kaggle/input/medical-image


In [4]:
# Medical Image Classification (Pneumonia Detection) with Overfitting Control

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

# ---------------------------
# Device (GPU if available)
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# 1. Data Augmentation
# ---------------------------
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ---------------------------
# 2. Load Dataset
# ---------------------------
dataset_path = "/root/.cache/kagglehub/datasets/bl9cksheep/medical-image/versions/3"

dataset = datasets.ImageFolder(dataset_path, transform=train_transform)

# Number of classes
num_classes = len(dataset.classes)
print("Classes:", dataset.classes)

# ---------------------------
# 3. Train / Validation Split
# ---------------------------
indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

# Different transforms for train and validation
train_subset.dataset.transform = train_transform
val_subset.dataset.transform = val_transform

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32)

# ---------------------------
# 4. Transfer Learning (ResNet18)
# ---------------------------
model = resnet18(weights=ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, num_classes)
)

model = model.to(device)

# ---------------------------
# 5. Loss and Optimizer
# ---------------------------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4   # L2 Regularization
)

# ---------------------------
# 6. Training with Early Stopping
# ---------------------------
best_val_loss = float("inf")
patience = 5
counter = 0
epochs = 20

for epoch in range(epochs):

    # -------- Training --------
    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # -------- Validation --------
    model.eval()
    val_loss = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # -------- Early Stopping --------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

    else:

        counter += 1

        if counter >= patience:
            print("Early stopping triggered")
            break

Classes: ['data', 'enhanced_data', 'test']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 141MB/s] 


Epoch 1 | Train Loss: 1.8824 | Val Loss: 0.8988
Epoch 2 | Train Loss: 0.5162 | Val Loss: 6.9748
Epoch 3 | Train Loss: 0.1430 | Val Loss: 0.6082
Epoch 4 | Train Loss: 0.1392 | Val Loss: 0.4673
Epoch 5 | Train Loss: 0.0794 | Val Loss: 0.5241
Epoch 6 | Train Loss: 0.0455 | Val Loss: 0.7638
Epoch 7 | Train Loss: 0.0608 | Val Loss: 1.3092
Epoch 8 | Train Loss: 0.0255 | Val Loss: 1.1145
Epoch 9 | Train Loss: 0.2251 | Val Loss: 0.7345
Early stopping triggered
